# 03/02 — Attenuation statistics per region

Three tests on pseudobulk:

* **T1 — interaction LFC**  : per-gene Welch on log-CPM, contrast BRI vs PBS.
* **T2 — sign concordance** : of disease-DEGs, fraction that reverse direction under BRICHOS (binomial test vs 50%).
* **T3 — attenuation slope**: regress LFC_residual on LFC_disease over disease-DEGs; slope <1 = rescue, with bootstrap CI.

Outputs `results/tables/attenuation/regional_attenuation_stats.tsv`.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 're_annotation_regions'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from utils.attenuation import (
    pseudobulk_lfc, sign_concordance, attenuation_slope
)

counts = pd.read_csv(TBL / 'pseudobulk_counts.tsv',
                     sep='\t', index_col=0)
meta   = pd.read_csv(TBL / 'pseudobulk_meta.tsv',
                     sep='\t', index_col=0)
regions = sorted(meta['region'].dropna().unique())
print(len(regions), 'regions')


### Per-region tests

In [ ]:
rows = []
padj_thresh = 0.05
for region in regions:
    try:
        pbs_v_wt = pseudobulk_lfc(counts, meta,
                                  group_a='PBS', group_b='WT',
                                  region=region)
        bri_v_wt = pseudobulk_lfc(counts, meta,
                                  group_a='BRICHOS', group_b='WT',
                                  region=region)
        bri_v_pbs = pseudobulk_lfc(counts, meta,
                                   group_a='BRICHOS', group_b='PBS',
                                   region=region)
    except ValueError as e:
        print(f'  {region}: skip ({e})')
        continue

    common = (pbs_v_wt.lfc.index
              .intersection(bri_v_wt.lfc.index)
              .intersection(bri_v_pbs.lfc.index))
    sig = pbs_v_wt.padj.loc[common] < padj_thresh

    sc_res = sign_concordance(pbs_v_wt.lfc.loc[common],
                              bri_v_pbs.lfc.loc[common],
                              sig)
    slope_res = attenuation_slope(pbs_v_wt.lfc.loc[common],
                                  bri_v_wt.lfc.loc[common],
                                  sig, n_boot=1000)

    rows.append(dict(
        region=region,
        n_disease_DEG=int(sig.sum()),
        n_reverse=sc_res['n_reverse'],
        frac_reverse=sc_res['fraction'],
        binomial_p=sc_res['p'],
        slope=slope_res['slope'],
        slope_ci_low=slope_res['ci_low'],
        slope_ci_high=slope_res['ci_high'],
        intercept=slope_res['intercept'],
        n_genes_in_slope=slope_res['n'],
    ))
stats_df = pd.DataFrame(rows).set_index('region').sort_values('slope')
stats_df


### Persist + plot

In [ ]:
stats_df.to_csv(TBL / 'regional_attenuation_stats.tsv', sep='\t')

fig, ax = plt.subplots(figsize=(5.5, 0.4 * len(stats_df) + 1.0))
y = np.arange(len(stats_df))
ax.errorbar(stats_df['slope'], y,
            xerr=[stats_df['slope'] - stats_df['slope_ci_low'],
                  stats_df['slope_ci_high'] - stats_df['slope']],
            fmt='o', color='#222', ecolor='#888', capsize=2)
ax.axvline(1, color='#bbb', linewidth=0.6, linestyle='--')
ax.axvline(0, color='#bbb', linewidth=0.6, linestyle=':')
ax.set_yticks(y); ax.set_yticklabels(stats_df.index, fontsize=8)
ax.set_xlabel('attenuation slope (LFC$_{BRI-WT}$ on LFC$_{PBS-WT}$)')
ax.set_title('1.0 = no rescue   |   0.0 = full rescue', fontsize=8,
             loc='left', color='#555')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / 'regional_attenuation_slope.svg',
            bbox_inches='tight')
fig.savefig(FIG / 'regional_attenuation_slope.png',
            bbox_inches='tight', dpi=200)
plt.show()
